# AskQE Pipeline - NLLB Backtranslation QA

This notebook runs Question Answering using **NLLB backtranslations**.
- Uses pre-generated questions from the baseline QG
- Uses NLLB backtranslations from `nllb_qg_merged.jsonl`
- **Run each language cell individually** - you can start with Russian and continue later

**Input:** `nllb_qg_merged.jsonl`  
**Output:** `results Qwen3B baseline/biomqm/nllb/QA/bt-{lang}-vanilla.jsonl`

## 0. Detect Environment & Configure Cache

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f'Environment: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    print(f'Model cache: {DRIVE_CACHE_DIR}')
elif IN_KAGGLE:
    KAGGLE_CACHE_DIR = '/kaggle/working/models_cache'
    os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = KAGGLE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(KAGGLE_CACHE_DIR, 'transformers')
    print(f'Model cache: {KAGGLE_CACHE_DIR}')

## 1. Setup - Install Dependencies & Clone Repository

In [ ]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'accelerate', 'nltk', 'sentence-transformers', 'sacrebleu', 'textstat'], check=True)

if IN_COLAB:
    if not os.path.exists('/content/askqe'):
        subprocess.run(['git', 'clone', 'https://github.com/laurabon/AskQE_DNLP_2025-2026.git', '/content/askqe'], check=True)
    PROJECT_ROOT = '/content/askqe'
elif IN_KAGGLE:
    if not os.path.exists('/kaggle/working/askqe'):
        subprocess.run(['git', 'clone', 'https://github.com/laurabon/AskQE_DNLP_2025-2026.git', '/kaggle/working/askqe'], check=True)
    PROJECT_ROOT = '/kaggle/working/askqe'
else:
    PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

## 2. Define Paths

In [ ]:
MERGED_INPUT = os.path.join(RESULTS_DIR, 'backtranslation', 'nllb_qg_merged.jsonl')
QA_SCRIPT = os.path.join(RESULTS_DIR, 'biomqm', 'direct-prompting', 'code', 'qwen-3b-direct-prompting.py')
OUTPUT_DIR = os.path.join(RESULTS_DIR, 'biomqm', 'nllb', 'QA')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Input: {MERGED_INPUT}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'\n✓ Input exists: {os.path.exists(MERGED_INPUT)}')

## 3. Pre-download Qwen Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('✓ Model cached')

## 4. Verify Input Data

In [ ]:
import json
from collections import Counter

with open(MERGED_INPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()

lang_counts = Counter(json.loads(line).get('lang_tgt', '?') for line in lines)
print(f'Total: {len(lines)} rows\n')
for lang, count in sorted(lang_counts.items()):
    print(f'  {lang}: {count}')

---

## 5. Question Answering

Run each language cell individually. Start with Russian, then continue with others later.

In [ ]:
# SOURCE QA (English original)
print('=== Source QA ===')
output = os.path.join(OUTPUT_DIR, 'source-vanilla.jsonl')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'source', '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

### 🇷🇺 Russian (ru)

In [ ]:
# RUSSIAN
lang = 'ru'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

### 🇩🇪 German (de)

In [ ]:
# GERMAN
lang = 'de'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

### 🇪🇸 Spanish (es)

In [ ]:
# SPANISH
lang = 'es'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

### 🇫🇷 French (fr)

In [ ]:
# FRENCH
lang = 'fr'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

### 🇨🇳 Chinese (zh-CN)

In [ ]:
# CHINESE
lang = 'zh-CN'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

---

## 6. Check Results

In [ ]:
print('=== Output Files ===')
if os.path.exists(OUTPUT_DIR):
    for f in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
        print(f'  {f}: {size:.2f} MB')

## 7. Download (Kaggle)

In [ ]:
if IN_KAGGLE:
    import shutil
    shutil.make_archive('/kaggle/working/nllb_qa_results', 'zip', OUTPUT_DIR)
    print('✓ Results zipped - download from Output tab')